# Irrigation Need — Notebook 5: TabPFN
**Kaggle Playground S6E4**

TabPFN is a transformer pretrained on millions of synthetic tabular datasets.
It performs inference via meta-learning — no hyperparameter tuning needed.
Architecturally completely different from gradient boosting → ideal ensemble candidate.

Constraint: TabPFN has a ~10k row limit for training.
Strategy: stratified sample from train, evaluate on full OOF via repeated sampling.

Inputs:
- `Data/processed/train_unscaled.parquet`
- `Data/processed/test_unscaled.parquet`
- `Data/processed/target_map.pkl`

Outputs:
- `submissions/submission_tabpfn.csv`
- `models/tabpfn_oof_preds.parquet`
- `models/tabpfn_test_probs.parquet`

Clean feature set: 19 raw original features only (no formula-derived columns).
Leakage fix: logit/prob columns dropped.


## 0. Setup & Install

In [1]:
# Install TabPFN if not already installed
import subprocess
subprocess.run(['pip', 'install', 'tabpfn', '-q'], check=True)
print('TabPFN installed.')

TabPFN installed.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

from tabpfn import TabPFNClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder

pd.set_option('display.float_format', '{:.4f}'.format)
PALETTE  = {'Low': '#02C39A', 'Medium': '#F4A261', 'High': '#E63946'}
SEED     = 42
N_FOLDS  = 5
MAX_ROWS = 10000  # TabPFN row limit per fit

LGBM_OOF_BA = 0.9705
XGB_OOF_BA  = 0.9719

os.makedirs('../models', exist_ok=True)
os.makedirs('../submissions', exist_ok=True)
print('Setup complete.')

Setup complete.


## 1. Load Data

In [3]:
train = pd.read_parquet('../Data/processed/train_unscaled.parquet')
test  = pd.read_parquet('../Data/processed/test_unscaled.parquet')

with open('../Data/processed/target_map.pkl', 'rb') as f:
    tm = pickle.load(f)
target_inv = tm['inv']  # {0: 'Low', 1: 'Medium', 2: 'High'}

y      = train['Irrigation_Need'].values
X      = train.drop(columns=['Irrigation_Need'])
X_test = test.copy()

print(f'Train : {X.shape}')
print(f'Test  : {X_test.shape}')
print(f'Target: {pd.Series(y).value_counts().to_dict()}')

Train : (630000, 38)
Test  : (270000, 38)
Target: {0: 369917, 1: 239074, 2: 21009}


## 2. Clean Feature Set — 19 Raw Originals Only

In [4]:
# Drop ALL formula-derived columns — leakage fix
drop_cols = [
    'soil_lt_25', 'temp_gt_30', 'rain_lt_300', 'wind_gt_10',
    'CGS_Flowering', 'CGS_Harvest', 'CGS_Sowing', 'CGS_Vegetative',
    'mulch_no', 'mulch_yes',
    'logit_low', 'logit_medium', 'logit_high',
    'prob_low', 'prob_medium', 'prob_high',
    'rule_score', 'logit_margin',
    '_drop_in_v2'
]

FEATURE_COLS = [c for c in X.columns if c not in drop_cols]

print(f'Clean feature set: {len(FEATURE_COLS)} features')
print(FEATURE_COLS)

X_clean      = X[FEATURE_COLS].copy()
X_test_clean = X_test[FEATURE_COLS].copy()

Clean feature set: 19 features
['Soil_Type', 'Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Field_Area_hectare', 'Mulching_Used', 'Previous_Irrigation_mm', 'Region']


## 3. Encode Categoricals for TabPFN
TabPFN requires numeric input — encode categoricals with LabelEncoder.

In [5]:
cat_cols = X_clean.select_dtypes(include='object').columns.tolist()
print(f'Categorical columns to encode: {cat_cols}')

# Load existing encoders from notebook 2
with open('../Data/processed/label_encoders.pkl', 'rb') as f:
    label_encoders = pickle.load(f)

for col in cat_cols:
    if col in label_encoders:
        X_clean[col]      = label_encoders[col].transform(X_clean[col])
        X_test_clean[col] = label_encoders[col].transform(X_test_clean[col])
    else:
        # Fallback: fit fresh encoder
        le = LabelEncoder()
        combined = pd.concat([X_clean[col], X_test_clean[col]])
        le.fit(combined)
        X_clean[col]      = le.transform(X_clean[col])
        X_test_clean[col] = le.transform(X_test_clean[col])

print('Encoding complete.')
print(X_clean.dtypes)

Categorical columns to encode: []
Encoding complete.
Soil_Type                    int64
Soil_pH                    float64
Soil_Moisture              float64
Organic_Carbon             float64
Electrical_Conductivity    float64
Temperature_C              float64
Humidity                   float64
Rainfall_mm                float64
Sunlight_Hours             float64
Wind_Speed_kmh             float64
Crop_Type                    int64
Crop_Growth_Stage            int64
Season                       int64
Irrigation_Type              int64
Water_Source                 int64
Field_Area_hectare         float64
Mulching_Used                int64
Previous_Irrigation_mm     float64
Region                       int64
dtype: object


## 4. TabPFN — Strategy

TabPFN limitation: max ~10k training rows per fit.

Approach:
- 5-fold CV: each fold has ~504k train rows → sample 9k stratified
- Predict on full validation fold (126k rows) — TabPFN handles large inference
- Repeat sampling N_BAGS times per fold to reduce variance from the sample
- Average probabilities across bags


In [ ]:
N_BAGS       = 15   # number of stratified samples per fold
SAMPLE_SIZE  = 1000  # safely below TabPFN 10k limit

X_arr      = X_clean.values
X_test_arr = X_test_clean.values

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_preds  = np.zeros(len(X_arr), dtype=int)
oof_probs  = np.zeros((len(X_arr), 3))
test_probs = np.zeros((len(X_test_arr), 3))
fold_scores = []

print(f'TabPFN — {N_FOLDS} folds × {N_BAGS} bags = {N_FOLDS * N_BAGS} fits')
print(f'Sample size per bag: {SAMPLE_SIZE:,} rows\n')

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_arr, y)):
    X_tr, X_val = X_arr[tr_idx], X_arr[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    val_probs_bags  = np.zeros((len(val_idx), 3))
    test_probs_bags = np.zeros((len(X_test_arr), 3))

    for bag in range(N_BAGS):
        # Stratified sample within the training fold
        rng = np.random.RandomState(SEED + fold * 100 + bag)
        sample_idx = []
        for cls in np.unique(y_tr):
            cls_idx = np.where(y_tr == cls)[0]
            n_sample = int(SAMPLE_SIZE * (cls_idx.shape[0] / len(y_tr)))
            n_sample = max(n_sample, 10)  # minimum 10 per class
            chosen = rng.choice(cls_idx, size=min(n_sample, len(cls_idx)), replace=False)
            sample_idx.extend(chosen)

        sample_idx = np.array(sample_idx)
        X_sample = X_tr[sample_idx]
        y_sample = y_tr[sample_idx]

        model = TabPFNClassifier(device='cpu', ignore_pretraining_limits=True)
        model.fit(X_sample, y_sample)

        val_probs_bags  += model.predict_proba(X_val) / N_BAGS
        test_probs_bags += model.predict_proba(X_test_arr) / N_BAGS

    oof_probs[val_idx] = val_probs_bags
    oof_preds[val_idx] = np.argmax(val_probs_bags, axis=1)
    test_probs        += test_probs_bags / N_FOLDS

    fold_ba = balanced_accuracy_score(y_val, oof_preds[val_idx])
    fold_scores.append(fold_ba)
    print(f'  Fold {fold+1}: BA = {fold_ba:.4f}  (sample={len(sample_idx):,}, val={len(val_idx):,})')

oof_ba = balanced_accuracy_score(y, oof_preds)
print(f'\nOOF Balanced Accuracy : {oof_ba:.4f}')
print(f'Mean fold BA          : {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')
print(f'LightGBM OOF          : {LGBM_OOF_BA:.4f}')
print(f'XGBoost  OOF          : {XGB_OOF_BA:.4f}')
print(f'TabPFN vs LightGBM    : {oof_ba - LGBM_OOF_BA:+.4f}')

TabPFN — 5 folds × 15 bags = 75 fits
Sample size per bag: 1,000 rows



tabpfn-v2.5-classifier-v2.5_default.ckpt:   0%|          | 0.00/42.9M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

## 5. OOF Diagnostics

In [ ]:
print('=== OOF Classification Report ===')
print(classification_report(y, oof_preds, target_names=['Low', 'Medium', 'High']))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm     = confusion_matrix(y, oof_preds)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Purples',
            xticklabels=['Low','Medium','High'],
            yticklabels=['Low','Medium','High'], ax=axes[0])
axes[0].set_xlabel('Predicted', fontweight='bold')
axes[0].set_ylabel('Actual', fontweight='bold')
axes[0].set_title(f'OOF Confusion Matrix — % of true class\n(BA={oof_ba:.4f})', fontweight='bold')

axes[1].bar(range(1, N_FOLDS+1), fold_scores, color='#9B59B6', alpha=0.8)
axes[1].axhline(oof_ba,       color='#E63946', linestyle='--', linewidth=2,
                label=f'TabPFN OOF ({oof_ba:.4f})')
axes[1].axhline(LGBM_OOF_BA, color='#028090', linestyle='--', linewidth=1.5,
                label=f'LightGBM ({LGBM_OOF_BA:.4f})')
axes[1].axhline(XGB_OOF_BA,  color='#F4A261', linestyle='--', linewidth=1.5,
                label=f'XGBoost ({XGB_OOF_BA:.4f})')
axes[1].set_xlabel('Fold')
axes[1].set_ylabel('Balanced Accuracy')
axes[1].set_title('Per-Fold BA — All Models', fontweight='bold')
axes[1].legend(fontsize=8)
axes[1].set_ylim(min(fold_scores) - 0.01, 1.0)
plt.tight_layout()
plt.show()

## 6. Agreement Analysis — TabPFN vs Tree Models

In [ ]:
lgbm_oof = pd.read_parquet('../models/lgbm_oof_preds.parquet')
xgb_oof  = pd.read_parquet('../models/xgb_oof_preds.parquet')

lgbm_preds = lgbm_oof['oof_pred'].values
xgb_preds  = xgb_oof['oof_pred'].values

agree_lgbm = (lgbm_preds == oof_preds).mean()
agree_xgb  = (xgb_preds  == oof_preds).mean()
agree_all  = ((lgbm_preds == oof_preds) & (xgb_preds == oof_preds)).mean()

print(f'TabPFN vs LightGBM agreement : {agree_lgbm:.4f} ({agree_lgbm*100:.1f}%)')
print(f'TabPFN vs XGBoost  agreement : {agree_xgb:.4f} ({agree_xgb*100:.1f}%)')
print(f'All 3 models agree           : {agree_all:.4f} ({agree_all*100:.1f}%)')

# Where TabPFN disagrees with both tree models
tabpfn_alone = (~(lgbm_preds == oof_preds)) & (~(xgb_preds == oof_preds))
print(f'\nTabPFN disagrees with BOTH trees: {tabpfn_alone.sum():,} rows ({tabpfn_alone.mean()*100:.2f}%)')
print('True class on those rows:')
print(pd.Series(y[tabpfn_alone]).map(target_inv).value_counts())

tabpfn_right_trees_wrong = tabpfn_alone & (oof_preds == y)
print(f'\nOf those: TabPFN correct, both trees wrong: {tabpfn_right_trees_wrong.sum():,} rows')
print('→ These are the rows where TabPFN adds unique value to the ensemble')

## 7. Prediction Distribution Check

In [ ]:
test_preds  = np.argmax(test_probs, axis=1)
test_labels = pd.Series(test_preds).map(target_inv)
oof_labels  = pd.Series(oof_preds).map(target_inv)
true_labels = pd.Series(y).map(target_inv)

order = ['Low', 'Medium', 'High']
print('=== Prediction Distribution Sanity Check ===')
for name, series in [('Train true', true_labels),
                     ('OOF preds',  oof_labels),
                     ('Test preds', test_labels)]:
    vc = series.value_counts(normalize=True).reindex(order).mul(100).round(1)
    print(f'  {name:12s}: Low {vc["Low"]}%  Medium {vc["Medium"]}%  High {vc["High"]}%')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors_bar = [PALETTE[c] for c in order]
for ax, (title, series) in zip(axes, [
    ('Train — true',    true_labels),
    ('OOF predictions', oof_labels),
    ('Test predictions', test_labels),
]):
    vc = series.value_counts(normalize=True).reindex(order) * 100
    ax.bar(order, vc.values, color=colors_bar, edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('%')
    ax.set_ylim(0, 80)
    for i, v in enumerate(vc.values):
        ax.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=10)

plt.suptitle('Prediction Distribution — Sanity Check', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Save Outputs

In [ ]:
# Submission
submission = pd.DataFrame({
    'id': X_test.index,
    'Irrigation_Need': test_labels.values
})
submission.to_csv('../submissions/submission_tabpfn.csv', index=False)
print('Saved: submissions/submission_tabpfn.csv')
print(submission['Irrigation_Need'].value_counts())

# OOF predictions for ensemble
oof_df = pd.DataFrame({
    'y_true':          y,
    'oof_pred':        oof_preds,
    'oof_prob_low':    oof_probs[:, 0],
    'oof_prob_medium': oof_probs[:, 1],
    'oof_prob_high':   oof_probs[:, 2],
}, index=X.index)
oof_df.to_parquet('../models/tabpfn_oof_preds.parquet')
print('Saved: models/tabpfn_oof_preds.parquet')

# Test probabilities for ensemble
test_probs_df = pd.DataFrame(
    test_probs,
    columns=['tabpfn_prob_low', 'tabpfn_prob_medium', 'tabpfn_prob_high'],
    index=X_test.index
)
test_probs_df.to_parquet('../models/tabpfn_test_probs.parquet')
print('Saved: models/tabpfn_test_probs.parquet')

## 9. Summary

In [ ]:
print('=' * 65)
print('TABPFN SUMMARY')
print('=' * 65)
print(f"""
APPROACH
  Model         : TabPFN (pretrained transformer for tabular data)
  Training rows : {SAMPLE_SIZE:,} stratified sample per bag
  Bags per fold : {N_BAGS}
  Total fits    : {N_FOLDS * N_BAGS}

FEATURE SET
  {len(FEATURE_COLS)} raw original features
  All formula-derived columns dropped (leakage fix)

RESULTS
  OOF Balanced Accuracy : {oof_ba:.4f}
  Fold scores           : {[round(s,4) for s in fold_scores]}
  Std across folds      : {np.std(fold_scores):.4f}

MODEL COMPARISON
  LightGBM OOF          : {LGBM_OOF_BA:.4f}
  XGBoost  OOF          : {XGB_OOF_BA:.4f}
  TabPFN   OOF          : {oof_ba:.4f}

DIVERSITY
  Agreement with LightGBM : {agree_lgbm*100:.1f}%
  Agreement with XGBoost  : {agree_xgb*100:.1f}%
  All 3 agree             : {agree_all*100:.1f}%
  Unique correct rows     : {tabpfn_right_trees_wrong.sum():,}

OUTPUTS
  submissions/submission_tabpfn.csv
  models/tabpfn_oof_preds.parquet
  models/tabpfn_test_probs.parquet
""")
print('=' * 65)
print('→ Proceed to 06_Ensemble.ipynb')
print('=' * 65)